libraries

In [2]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os
from dotenv import load_dotenv
from groq import Groq
import PyPDF2
from sentence_transformers import SentenceTransformer
import faiss


load_dotenv()
client = Groq(api_key=os.getenv("api_key"))
print("Libraries loaded!")


c:\Users\yasha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 1


Libraries loaded!


Dataset load

In [3]:
df = pd.read_csv("intents_large.csv")
print("Data loaded!")
print(df["intent"].value_counts())

Data loaded!
intent
greeting        40
bye             40
price           40
help            40
order           40
order_status    40
return          40
hours           40
location        40
thanks          40
about           40
Name: count, dtype: int64


In [5]:
def load_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

pdf_text = load_pdf("products.pdf")
print(f"Total characters: {len(pdf_text)}")
print("\nPDF first 500 characters:")
print(pdf_text[:500])

Total characters: 2100

PDF first 500 characters:
ShopBot - Product Catalog
MG Road, Bhopal | Mon-Sat 9AM-6PM | Returns within 7 days
  Smartphones
  - iPhone 15 Rs 79,999
  - Samsung Galaxy S24 Rs 74,999
  - OnePlus 12 Rs 64,999
  - Redmi Note 13 Pro Rs 26,999
  - Realme 12 Pro Rs 22,999
  - Vivo V30 Rs 34,999
  - Oppo Reno 11 Rs 29,999
  - Google Pixel 8 Rs 75,999
  Laptops
  - MacBook Air M2 Rs 1,14,999
  - Dell Inspiron 15 Rs 55,999
  - HP Pavilion 14 Rs 49,999
  - Lenovo IdeaPad Slim 5 Rs 52,999
  - Asus VivoBook 15 Rs 47,999
  - Acer Aspi


In [7]:
def create_chunks(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start = end - overlap
    return chunks

chunks = create_chunks(pdf_text)
print(f"Total chunks: {len(chunks)}")
print("\nfirst chunk:")
print(chunks[0])
print("\nSecond chunk:")
print(chunks[1])

Total chunks: 14

first chunk:
ShopBot - Product Catalog
MG Road, Bhopal | Mon-Sat 9AM-6PM | Returns within 7 days
  Smartphones
  - iPhone 15 Rs 79,999
  - Samsung Galaxy S24 Rs 74,999
  - OnePlus 12 Rs 64,999
  - Redmi Note 13 Pr

Second chunk:
,999
  - OnePlus 12 Rs 64,999
  - Redmi Note 13 Pro Rs 26,999
  - Realme 12 Pro Rs 22,999
  - Vivo V30 Rs 34,999
  - Oppo Reno 11 Rs 29,999
  - Google Pixel 8 Rs 75,999
  Laptops
  - MacBook Air M2 Rs


In [8]:
X = df["text"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorizer ready!")

Vectorizer ready!


Models

In [10]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline                            #Naive Bayes
from sklearn.feature_extraction.text import TfidfVectorizer

nb_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', MultinomialNB(alpha=0.1))
])

nb_improved.fit(X_train, y_train)
nb_imp_preds = nb_improved.predict(X_test)
print(f" Naive Bayes: {accuracy_score(y_test, nb_imp_preds) * 100:.1f}%")

 Naive Bayes: 85.2%


In [11]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', LogisticRegression(max_iter=500, C=10))
])

lr_improved.fit(X_train, y_train)
lr_imp_preds = lr_improved.predict(X_test)
print(f" Logistic Regression: {accuracy_score(y_test, lr_imp_preds) * 100:.1f}%")

 Logistic Regression: 80.7%


In [12]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', RandomForestClassifier(n_estimators=200, max_depth=None))
])

rf_improved.fit(X_train, y_train)
rf_imp_preds = rf_improved.predict(X_test)
print(f" Random Forest: {accuracy_score(y_test, rf_imp_preds) * 100:.1f}%")

 Random Forest: 68.2%


In [13]:
#LinearSVC
from sklearn.svm import LinearSVC

svc_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', LinearSVC(C=10, max_iter=2000))
])

svc_improved.fit(X_train, y_train)
svc_imp_preds = svc_improved.predict(X_test)
print(f"Improved LinearSVC: {accuracy_score(y_test, svc_imp_preds) * 100:.1f}%")

Improved LinearSVC: 83.0%


In [14]:

results = {
    "Naive Bayes":          accuracy_score(y_test, nb_imp_preds),
    "Logistic Regression":  accuracy_score(y_test, lr_imp_preds),
    "Random Forest":        accuracy_score(y_test, rf_imp_preds),
    "LinearSVC":            accuracy_score(y_test, svc_imp_preds),
}

best_name = max(results, key=results.get)
best_models = {
    "Naive Bayes":         nb_improved,
    "Logistic Regression": lr_improved,
    "Random Forest":       rf_improved,
    "LinearSVC":           svc_improved,
}

best_model = best_models[best_name]
print(f"Best model: {best_name} ({results[best_name]*100:.1f}%)")

with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved!")

Best model: Naive Bayes (85.2%)
Best model saved!


In [15]:
from groq import Groq

from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("api_key"))

def get_response(user_input):
    intent = best_model.predict([user_input.lower()])[0]

    prompt = f"""You are a helpful customer support chatbot for a shop in Bhopal.
Customer said: "{user_input}"
Detected intent: "{intent}"
Reply in a short, friendly, natural way (2-3 sentences max).
Shop details:
- Pricing: Basic Rs 499, Pro Rs 999, Premium Rs 1999
- Returns within 7 days, refund in 3-5 days
- Open Monday to Saturday, 9 AM to 6 PM
- Located at MG Road, Bhopal"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )

    reply = response.choices[0].message.content
    return reply, intent

print("Chatbot ready! Type 'quit' to stop.\n")

while True:
    user = input("You: ").strip()
    if user.lower() == "quit":
        print("Bot: Goodbye!")
        break
    if not user:
        continue
    reply, intent = get_response(user)
    print(f"Bot [{intent}]: {reply}\n")

python-dotenv could not parse statement starting at line 1


Chatbot ready! Type 'quit' to stop.

Bot: Goodbye!


In [16]:
def create_chunks(text, chunk_size=200, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        start = end - overlap
    return chunks

chunks = create_chunks(pdf_text)
print(f"Total chunks: {len(chunks)}")

Total chunks: 14


In [17]:
# Load the sentence transformer model
print("Loading model — please wait...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Convert chunks to embeddings
embeddings = embedder.encode(chunks)
embeddings = np.array(embeddings).astype("float32")

print(f"Embeddings shape: {embeddings.shape}")
print("Embeddings ready!")

Loading model — please wait...


c:\Users\yasha\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yasha\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6441.23it/s]
BertMo

Embeddings shape: (14, 384)
Embeddings ready!


In [18]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Total chunks in FAISS index: {index.ntotal}")
print("FAISS ready!")

Total chunks in FAISS index: 14
FAISS ready!


In [19]:
# Function to search relevant chunks from PDF
def search_chunks(query, top_k=3):
    query_embedding = embedder.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    results = [chunks[i] for i in indices[0]]
    return results

# Test the search
test = search_chunks("iPhone price")
print("Search result:")
print(test[0])

Search result:
 Rs 35,999
  - Fujifilm Instax Mini 12 Rs 7,499
  - DJI Osmo Pocket 3 Rs 44,999
  Accessories
  - Anker 65W USB-C Charger Rs 2,499
  - Samsung 25W Fast Charger Rs 1,499
  - Logitech MX Master 3 Mouse 


In [21]:
# RAG chatbot — answers from PDF using Groq

# Direct API key (temporary fix)
client = Groq(api_key="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")

def rag_response(user_input):
    # Find relevant chunks from PDF
    relevant_chunks = search_chunks(user_input, top_k=3)
    context = "\n".join(relevant_chunks)

    # Send context + question to Groq
    prompt = f"""You are a helpful chatbot for ShopBot store in Bhopal.
Use the following information from our product catalog to answer the question.

Product Catalog Information:
{context}

Customer Question: {user_input}

Answer in a short friendly way (2-3 sentences).
If the answer is not in the catalog, say "I don't have that information."
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )

    return response.choices[0].message.content

print("RAG Chatbot is ready! Type 'quit' to stop.\n")

while True:
    user = input("You: ").strip()
    if user.lower() == "quit":
        print("Bot: Goodbye!")
        break
    if not user:
        continue
    reply = rag_response(user)
    print(f"Bot: {reply}\n")

RAG Chatbot is ready! Type 'quit' to stop.

Bot: The MacBook Air M2 is available at our store for Rs 1,14,999. If you're looking for other MacBook models, I don't have that information in our current catalog. Would you like to know more about the MacBook Air M2 or explore other laptop options?

Bot: Goodbye!
